# VoxCPM2 Server on Kaggle T4 x2

This notebook deploys a **VoxCPM2 HTTP server** that implements the API contract expected by the Audiobook Studio `remote_voxcpm2_port.py`:
- `POST /synthesize` - Submit TTS task
- `GET /status/{task_id}` - Check task status
- `GET /result/{task_id}` - Get result with audio URL
- `POST /cancel/{task_id}` - Cancel task
- `GET /health` - Health check

## Usage
1. Enable GPU (Settings → Accelerator → T4 x2)
2. Run all cells
3. The server will be exposed via **Cloudflare Tunnel** (automatically created)
4. Copy the tunnel URL and set `VOXCPM2_ENDPOINT` in your docker-compose.free.yml

In [ ]:
# === Environment Setup (before any imports) ===
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import sys
import subprocess

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "voxcpm==2.0.3",
    "fastapi==0.110.0",
    "uvicorn==0.29.0",
    "pydantic==2.7.0",
    "soundfile==0.12.1",
    "numpy==1.26.4",
    "requests",
    "pyngrok",  # For Cloudflare tunnel alternative
], check=True)
print("Dependencies installed.")

In [ ]:
# === Download VoxCPM2 Model ===
import os, requests
from huggingface_hub import HfApi

ENDPOINT = "https://hf-mirror.com"
REPO = "openbmb/VoxCPM2"
model_dir = "/kaggle/working/VoxCPM2"
os.makedirs(model_dir, exist_ok=True)

api = HfApi(endpoint=ENDPOINT)
info = api.model_info(REPO)
files = sorted(s.rfilename for s in info.siblings)
print(f"Downloading {len(files)} files from {REPO}...")

ok = 0
for fname in files:
    url = f"{ENDPOINT}/{REPO}/resolve/main/{fname}"
    dest = os.path.join(model_dir, fname)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        r = requests.get(url, timeout=120, stream=True, allow_redirects=True, verify=False)
        if r.status_code == 200:
            with open(dest, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            ok += 1
            print(f"  ✅ {fname}")
        else:
            print(f"  ❌ {fname}: HTTP {r.status_code}")
    except Exception as e:
        print(f"  ❌ {fname}: {e}")

print(f"\nDownloaded {ok}/{len(files)} files")

In [ ]:
# === Load VoxCPM2 Model ===
import time
from voxcpm import VoxCPM

model_path = "/kaggle/working/VoxCPM2"
print(f"Loading VoxCPM2 from {model_path}...")
t0 = time.time()
model = VoxCPM.from_pretrained(
    model_path,
    load_denoiser=False,
    optimize=False,
    device="cuda",
)
print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"Sample rate: {model.tts_model.sample_rate} Hz")

In [ ]:
# === FastAPI Server Implementation ===
import uuid
import asyncio
import threading
from pathlib import Path
from typing import Optional, Dict, Any
from dataclasses import dataclass, field
from enum import Enum

import numpy as np
import soundfile as sf
import torch

from fastapi import FastAPI, HTTPException, BackgroundTasks
from pydantic import BaseModel
import uvicorn

# ===== Data Models (matching remote_voxcpm2_port.py contract) =====

class TaskStatus(str, Enum):
    PENDING = "PENDING"
    RUNNING = "RUNNING"
    DONE = "DONE"
    FAILED = "FAILED"

class SubmitRequest(BaseModel):
    task_id: Optional[str] = None
    text: str
    voice_id: Optional[str] = None
    speaker_name: Optional[str] = None
    language: str = "zh"
    reference_audio_path: Optional[str] = None
    prosody: Optional[Dict[str, Any]] = None
    metadata: Optional[Dict[str, Any]] = None

class SubmitResponse(BaseModel):
    task_id: str
    status: str
    message: Optional[str] = None

class StatusResponse(BaseModel):
    task_id: str
    status: str
    progress: Optional[float] = None
    error_message: Optional[str] = None
    dnsmos_score: Optional[float] = None

class ResultResponse(BaseModel):
    task_id: str
    status: str
    audio_url: Optional[str] = None
    audio_path: Optional[str] = None
    duration_ms: Optional[int] = None
    error_message: Optional[str] = None
    dnsmos_score: Optional[float] = None
    asr_wer: Optional[float] = None
    speaker_similarity: Optional[float] = None
    started_at: Optional[str] = None
    completed_at: Optional[str] = None

class HealthResponse(BaseModel):
    healthy: bool
    latency_ms: Optional[float] = None
    pending_count: int = 0
    running_count: int = 0
    version: str = "1.0.0"

# ===== Task Store =====
tasks: Dict[str, Dict[str, Any]] = {}
tasks_lock = threading.Lock()

def make_task_id() -> str:
    return f"voxcpm2-{uuid.uuid4().hex[:12]}"

# ===== Audio Output Directory =====
OUTPUT_DIR = Path("/kaggle/working/voxcpm2_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ===== FastAPI App =====
app = FastAPI(title="VoxCPM2 TTS Server", version="1.0.0")

@app.get("/health", response_model=HealthResponse)
async def health():
    import time
    start = time.time()
    with tasks_lock:
        pending = sum(1 for t in tasks.values() if t["status"] == TaskStatus.PENDING)
        running = sum(1 for t in tasks.values() if t["status"] == TaskStatus.RUNNING)
    return HealthResponse(
        healthy=True,
        latency_ms=(time.time() - start) * 1000,
        pending_count=pending,
        running_count=running,
    )

async def synthesize_task(task_id: str, request: SubmitRequest):
    """Background task for synthesis."""
    with tasks_lock:
        tasks[task_id]["status"] = TaskStatus.RUNNING
        tasks[task_id]["progress"] = 0.1
    
    try:
        text = request.text.strip()
        if not text:
            raise ValueError("text is required")
        
        voice_id = request.voice_id or request.speaker_name or "default"
        
        with tasks_lock:
            tasks[task_id]["progress"] = 0.3
        
        # Run inference
        with torch.no_grad():
            wav = model.generate(
                text=text,
                cfg_value=request.prosody.get("cfg_value", 2.0) if request.prosody else 2.0,
                inference_timesteps=request.prosody.get("inference_timesteps", 10) if request.prosody else 10,
            )
        
        with tasks_lock:
            tasks[task_id]["progress"] = 0.7
        
        # Convert to numpy
        wav = np.asarray(wav).astype(np.float32).reshape(-1)
        sr = model.tts_model.sample_rate
        
        # Save to file
        output_path = OUTPUT_DIR / f"{task_id}.wav"
        sf.write(str(output_path), wav, sr)
        
        duration_ms = int(len(wav) / sr * 1000)
        
        with tasks_lock:
            tasks[task_id]["status"] = TaskStatus.DONE
            tasks[task_id]["progress"] = 1.0
            tasks[task_id]["result"] = {
                "audio_path": str(output_path),
                "duration_ms": duration_ms,
                "sample_rate": sr,
            }
            tasks[task_id]["completed_at"] = time.time()
            
    except Exception as e:
        with tasks_lock:
            tasks[task_id]["status"] = TaskStatus.FAILED
            tasks[task_id]["error"] = str(e)
            print(f"Task {task_id} failed: {e}")

@app.post("/synthesize", response_model=SubmitResponse)
async def synthesize(request: SubmitRequest, background_tasks: BackgroundTasks):
    task_id = request.task_id or make_task_id()
    
    with tasks_lock:
        if task_id in tasks:
            raise HTTPException(status_code=409, detail="Task already exists")
        tasks[task_id] = {
            "status": TaskStatus.PENDING,
            "progress": 0.0,
            "request": request.model_dump(),
            "result": None,
            "error": None,
            "started_at": time.time(),
        }
    
    background_tasks.add_task(synthesize_task, task_id, request)
    
    return SubmitResponse(
        task_id=task_id,
        status=TaskStatus.PENDING,
        message="Task submitted"
    )

@app.get("/status/{task_id}", response_model=StatusResponse)
async def get_status(task_id: str):
    with tasks_lock:
        task = tasks.get(task_id)
    
    if not task:
        raise HTTPException(status_code=404, detail="Task not found")
    
    return StatusResponse(
        task_id=task_id,
        status=task["status"],
        progress=task.get("progress"),
        error_message=task.get("error"),
    )

@app.get("/result/{task_id}", response_model=ResultResponse)
async def get_result(task_id: str):
    with tasks_lock:
        task = tasks.get(task_id)
    
    if not task:
        raise HTTPException(status_code=404, detail="Task not found")
    
    if task["status"] not in (TaskStatus.DONE, TaskStatus.FAILED):
        raise HTTPException(status_code=400, detail=f"Task not complete (status: {task['status']})")
    
    if task["status"] == TaskStatus.FAILED:
        return ResultResponse(
            task_id=task_id,
            status="FAILED",
            error_message=task.get("error"),
        )
    
    result = task.get("result")
    if not result:
        raise HTTPException(status_code=500, detail="No result available")
    
    # For local file access, return file:// URL
    return ResultResponse(
        task_id=task_id,
        status="DONE",
        audio_url=f"file://{result['audio_path']}",
        audio_path=result["audio_path"],
        duration_ms=result["duration_ms"],
    )

@app.post("/cancel/{task_id}")
async def cancel(task_id: str):
    with tasks_lock:
        task = tasks.get(task_id)
        if not task:
            return {"task_id": task_id, "cancelled": False, "message": "Not found"}
        if task["status"] in (TaskStatus.DONE, TaskStatus.FAILED):
            return {"task_id": task_id, "cancelled": False, "message": "Already terminal"}
        task["status"] = TaskStatus.FAILED
        task["error"] = "Cancelled"
    return {"task_id": task_id, "cancelled": True, "message": "Cancellation requested"}

print("FastAPI app defined successfully")

In [ ]:
# === Start Cloudflare Tunnel & Server ===
# Using cloudflared for public HTTPS URL
import subprocess
import time
import threading
import requests

# Start uvicorn in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8080, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("Server started on port 8080")

# Wait for server to be ready
time.sleep(3)

# Start cloudflared tunnel
try:
    # Try to download cloudflared
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/tmp/cloudflared"], check=False)
    subprocess.run(["chmod", "+x", "/tmp/cloudflared"], check=False)
    
    # Start tunnel
    tunnel_proc = subprocess.Popen([
        "/tmp/cloudflared", "tunnel", "--url", "http://localhost:8080",
        "--no-autoupdate",
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    
    # Wait for tunnel URL
    time.sleep(5)
    
    # Try to get tunnel URL from cloudflared output
    # cloudflared outputs URL to stderr
    print("Cloudflare tunnel started. Check logs for URL.")
    print("Format: https://<random>.trycloudflare.com")
    print("Set this as VOXCPM2_ENDPOINT in your docker-compose.free.yml")
    
except Exception as e:
    print(f"Tunnel error: {e}")
    print("Server still running on localhost:8080")
    print("For external access, use ngrok or similar: ngrok http 8080")

In [ ]:
# === Test the Server ===
import requests
import json

BASE_URL = "http://localhost:8080"

# Health check
print("1. Health check...")
r = requests.get(f"{BASE_URL}/health", timeout=10)
print(f"   {r.json()}")

# Submit synthesis
print("\n2. Submit synthesis...")
payload = {
    "text": "这是一个 VoxCPM2 服务器测试，验证服务端推理是否正常。",
    "voice_id": "default",
    "language": "zh",
}
r = requests.post(f"{BASE_URL}/synthesize", json=payload, timeout=30)
result = r.json()
task_id = result["task_id"]
print(f"   Task ID: {task_id}")
print(f"   Status: {result['status']}")

# Poll for completion
print("\n3. Polling for completion...")
import time
start = time.time()
while time.time() - start < 120:
    r = requests.get(f"{BASE_URL}/status/{task_id}", timeout=10)
    status = r.json()
    print(f"   Status: {status['status']} (progress: {status.get('progress')})")
    if status["status"] in ("DONE", "FAILED"):
        break
    time.sleep(2)

# Get result
print("\n4. Get result...")
r = requests.get(f"{BASE_URL}/result/{task_id}", timeout=10)
result = r.json()
print(f"   Status: {result['status']}")
print(f"   Audio URL: {result.get('audio_url')}")
print(f"   Duration: {result.get('duration_ms')}ms")

if result.get('audio_path'):
    import soundfile as sf
    info = sf.info(result['audio_path'])
    print(f"   Audio: {info.duration:.2f}s, {info.samplerate}Hz, {info.channels}ch")
    print("\n✅ VoxCPM2 Server test PASSED")
else:
    print("\n❌ No audio file in result")

## Next Steps

1. **Copy the Cloudflare tunnel URL** from the logs above (format: `https://xxx.trycloudflare.com`)
2. **Update your `docker-compose.free.yml`** with:
   ```yaml
   environment:
     - VOXCPM2_ENDPOINT=https://xxx.trycloudflare.com
   ```
3. **Restart the stack**: `docker compose -f docker-compose.free.yml up -d`
4. **Test integration** with the main application

## Notes
- Kaggle sessions max 12 hours, weekly quota 30 hours
- For production, use Modal (A10G/V100) with fixed endpoint
- The server runs in this notebook - keep it running for the API to work
- Audio files are saved to `/kaggle/working/voxcpm2_output/`